In [1]:
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import re, json, io
import pandas as pd
import spacy
from spacy import displacy
nlp = spacy.load("en_core_web_sm")

In [3]:
text = """Apple Inc. announced on 12 September 2025 that Tim Cook will open a new office in Bengaluru, India.
The company will invest $500 million over the next 3 years. Contact the press team at press@apple.com
or call +91 98765 43210. Sundar Pichai of Google met Narendra Modi in New Delhi on 3 March 2025."""
doc = nlp(text)

In [4]:
for ent in doc.ents:
    print(f"{ent.text:25s} -> {ent.label_:10s} ({spacy.explain(ent.label_)})")
displacy.render(doc, style="ent", jupyter=True)

Apple Inc.                -> ORG        (Companies, agencies, institutions, etc.)
12 September 2025         -> DATE       (Absolute or relative dates or periods)
Tim Cook                  -> PERSON     (People, including fictional)
Bengaluru                 -> GPE        (Countries, cities, states)
India                     -> GPE        (Countries, cities, states)
$500 million              -> MONEY      (Monetary values, including unit)
the next 3 years          -> DATE       (Absolute or relative dates or periods)
98765 43210               -> DATE       (Absolute or relative dates or periods)
Sundar Pichai             -> PERSON     (People, including fictional)
Google                    -> ORG        (Companies, agencies, institutions, etc.)
Narendra Modi             -> PERSON     (People, including fictional)
New Delhi                 -> GPE        (Countries, cities, states)
3 March 2025              -> DATE       (Absolute or relative dates or periods)


In [5]:
patterns = {
    "email": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone": r"\+?\d{2}[\s-]?\d{5}[\s-]?\d{5}",
    "date":  r"\b\d{1,2}\s(?:January|February|March|April|May|June|July|August|September|October|November|December)\s\d{4}\b",
    "money": r"\$\d[\d,]*(?:\.\d+)?\s?(?:million|billion)?",
}
for name, pat in patterns.items():
    print(f"{name:6s}:", re.findall(pat, text))

email : ['press@apple.com']
phone : ['+91 98765 43210']
date  : ['12 September 2025', '3 March 2025']
money : ['$500 million']


In [6]:
def svo(doc):
    triples = []
    for tok in doc:
        if tok.pos_ == "VERB":
            subj = [w for w in tok.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w for w in tok.rights if w.dep_ in ("dobj", "attr")]
            for s in subj:
                for o in obj:
                    triples.append((" ".join(t.text for t in s.subtree), tok.lemma_, " ".join(t.text for t in o.subtree)))
    return triples

for t in svo(doc): print(t)

('Tim Cook', 'open', 'a new office in Bengaluru , India')
('The company', 'invest', '$ 500 million')
('Sundar Pichai of Google', 'meet', 'Narendra Modi')


In [7]:
record = {
    "organizations": sorted({e.text for e in doc.ents if e.label_ == "ORG"}),
    "people": sorted({e.text for e in doc.ents if e.label_ == "PERSON"}),
    "locations": sorted({e.text for e in doc.ents if e.label_ in ("GPE", "LOC")}),
    "dates": re.findall(patterns["date"], text),
    "emails": re.findall(patterns["email"], text),
    "money": re.findall(patterns["money"], text),
}
print(json.dumps(record, indent=2))

{
  "organizations": [
    "Apple Inc.",
    "Google"
  ],
  "people": [
    "Narendra Modi",
    "Sundar Pichai",
    "Tim Cook"
  ],
  "locations": [
    "Bengaluru",
    "India",
    "New Delhi"
  ],
  "dates": [
    "12 September 2025",
    "3 March 2025"
  ],
  "emails": [
    "press@apple.com"
  ],
  "money": [
    "$500 million"
  ]
}


In [8]:
csv_data = """id,name,email,joined,salary
1,Asha Rao,asha@example.com,2023-04-12,55000
2,Ravi Kumar,ravi@example.org,2022-11-03,62000
3,Meena Iyer,meena@example.com,2024-01-20,58000
"""
df = pd.read_csv(io.StringIO(csv_data), parse_dates=["joined"])
display(df)

print("Employees joined after 2023:", df[df.joined > "2023-01-01"].name.tolist())
print("Average salary :", df.salary.mean())
print("Email domains  :", df.email.str.extract(r"@(.+)$")[0].value_counts().to_dict())

js = json.loads('{"order": {"id": 101, "customer": {"name": "Ravi", "city": "Lucknow"}, "items": [{"sku": "A1", "qty": 2}, {"sku": "B7", "qty": 1}]}}')
print("\nJSON extraction ->", js["order"]["customer"]["name"], js["order"]["customer"]["city"],
      sum(i["qty"] for i in js["order"]["items"]), "items")

,id,name,email,joined,salary
0,1,Asha Rao,asha@example.com,2023-04-12,55000
1,2,Ravi Kumar,ravi@example.org,2022-11-03,62000
2,3,Meena Iyer,meena@example.com,2024-01-20,58000


Employees joined after 2023: ['Asha Rao', 'Meena Iyer']
Average salary : 58333.333333333336
Email domains  : {'example.com': 2, 'example.org': 1}

JSON extraction -> Ravi Lucknow 3 items
